# Version débile réseau
Débile car il faudrait virer les stop words avant, sans doute bosser à l'échelle de la phrase, gérer les risques d'apparition multiples de république dans une intervention avec parfois faux flag qui passe car vrai flag dans l'intervention, etc.

In [15]:
import pandas as pd
from itertools import combinations
import networkx as nx

df = pd.read_csv("context_counts_window_5.csv")
df = df.rename(columns={"Unnamed: 0": "context"})
df

,context,count
0,respect des principes de la République et de l...,16
1,de l école de la République,15
2,respect des principes de la République,8
3,l histoire de la Ve République,7
4,Constitution La France est une République indi...,6
...,...,...
19365,de la promesse républicaine La République doit...,1
19366,la réalité de la promesse républicaine La Répu...,1
19367,contre l extrémisme Si la République est affai...,1
19368,que la défense des principes républicains est ...,1


In [ ]:
from itertools import combinations
import networkx as nx
from ipysigma import Sigma
import stopwordsiso as stopwords
import re

# stopwords français
stop_fr = stopwords.stopwords("fr")

# regex pour la racine répu*
pattern_repu = re.compile(r"répu\w*", re.IGNORECASE)


G = nx.Graph()

for _, row in df.iterrows():
    tokens = row["context"].lower().split()

    tokens = [
        t
        for t in tokens
        if t not in stop_fr  # enlever stopwords
        and not pattern_repu.match(t)  # enlever répu*
        and len(t) > 2  # enlever mots très courts
    ]

    poids = row["count"]

    for w1, w2 in combinations(tokens, 2):
        if G.has_edge(w1, w2):
            G[w1][w2]["weight"] += poids
        else:
            G.add_edge(w1, w2, weight=poids)

edges_to_remove = [(u, v) for u, v, w in G.edges(data="weight") if w < 5]

G.remove_edges_from(edges_to_remove)

for n in G.nodes():
    G.nodes[n]["size"] = G.degree(n)
    G.nodes[n]["label"] = n

reseau = Sigma(G, node_size="size", edge_size="weight", node_label="label")
reseau


In [ ]:
# pour filtrer les nœuds très connectés, mais pas forcément pertinents

# if G.degree("principes") > 30:
#     G.remove_node("principes")

In [ ]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G))

for i, com in enumerate(communities):
    for node in com:
        G.nodes[node]["community"] = i

Sigma(
    G,
    node_size="size",
    edge_size="weight",
    node_color="community",
    node_label=lambda n: n,
)

Sigma(nx.Graph with 10,370 nodes and 850 edges)

## PMI

In [ ]:
import stopwordsiso as stopwords
import re
from collections import Counter
from itertools import combinations
import math

# stop et pattern regex pour filtrer
stop_fr = stopwords.stopwords("fr")
pattern_repu = re.compile(r"répu\w*", re.IGNORECASE)


def tokenize(text):
    tokens = text.lower().split()

    tokens = [
        t
        for t in tokens
        if t not in stop_fr and not pattern_repu.match(t) and len(t) > 2
    ]

    return tokens

# compter occurrences de mots et paires
word_counts = Counter()
pair_counts = Counter()
total_windows = 0

# parcourir chaque contexte et compter les mots et paires avec leur poids
for _, row in df.iterrows():
    tokens = tokenize(row["context"])
    weight = row["count"]

    unique_tokens = set(tokens)

    for w in unique_tokens:
        word_counts[w] += weight

    for w1, w2 in combinations(sorted(unique_tokens), 2):
        pair_counts[(w1, w2)] += weight

    total_windows += weight


# calculer PMI pour chaque paire
pmi_scores = {}

for (w1, w2), c12 in pair_counts.items():
    p12 = c12 / total_windows
    p1 = word_counts[w1] / total_windows
    p2 = word_counts[w2] / total_windows

    pmi = math.log2(p12 / (p1 * p2))

    if pmi > 0:  # garder seulement associations positives
        pmi_scores[(w1, w2)] = pmi

In [56]:
# Display a sample of PMI scores
list(pmi_scores.items())[:10]

# Or get count and statistics
print(f"Total pairs: {len(pmi_scores)}")
print(f"Min PMI: {min(pmi_scores.values())}")
print(f"Max PMI: {max(pmi_scores.values())}")
print(f"Mean PMI: {sum(pmi_scores.values()) / len(pmi_scores)}")

Total pairs: 67017
Min PMI: 0.001760457864437404
Max PMI: 14.264442600226602
Mean PMI: 6.199393861888268


In [70]:
import networkx as nx

G = nx.Graph()

for (w1, w2), score in pmi_scores.items():
    if score > 12:  # aviser sur seuil utile pour filtrer le bruit, bourrin
        G.add_edge(w1, w2, weight=score)

for node in G.nodes():
    G.nodes[node]["size"] = G.degree(node)
    G.nodes[node]["label"] = node

In [71]:
G.number_of_nodes(), G.number_of_edges()

(4605, 3469)

In [72]:
from ipysigma import Sigma

Sigma(G, node_size="size", edge_size="weight", node_label="label")

Sigma(nx.Graph with 4,605 nodes and 3,469 edges)

In [73]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G))

for i, com in enumerate(communities):
    for node in com:
        G.nodes[node]["community"] = i

reseau_PMI_commu = Sigma(
    G, node_size="size", edge_size="weight", node_color="community", node_label="label"
)

In [74]:
reseau_PMI_commu

Sigma(nx.Graph with 4,605 nodes and 3,469 edges)

# depuis df base

In [ ]:
import spacy

nlp = spacy.load("fr_core_news_sm")

def split_sentences(text):
    doc = nlp(text)
    return [sent.text for sent in doc.sents]

def extract_sentence_contexts(series):
    
    contexts = []

    for text in series.dropna():
        
        sentences = sentence_split.split(text)

        for sent in sentences:
            
            excl_positions = []
            
            excl_positions.extend(
                [m.span() for m in pattern_excl_case_sensitive.finditer(sent)]
            )
            
            excl_positions.extend(
                [m.span() for m in pattern_excl_case_insensitive.finditer(sent)]
            )

            def in_excl(pos):
                return any(start <= pos < end for start, end in excl_positions)

            for match in pattern_lexical.finditer(sent):
                pos = match.start()
                
                if not in_excl(pos):
                    contexts.append(sent.strip())
                    break   # une occurrence suffit pour garder la phrase

    return contexts

In [ ]:
contexts = extract_sentence_contexts(df["text"])

In [ ]:
contexts_df = (
    pd.Series(contexts)
    .value_counts()
    .reset_index()
)

contexts_df.columns = ["context", "count"]